# Anime Recommendation System

This notebook is the cleaned main pipeline for Team 12's anime recommendation project.

The project combines:

- CF: Surprise SVD over explicit user-anime ratings.
- CB: TF-IDF and metadata features from `anime.csv`.
- Meta learner: XGBoost, LightGBM, and CatBoost models that learn from `(cf_score, cb_score)`.

Large raw rating files live under `data/raw/` and are tracked with Git LFS. Precomputed metadata and meta-training files live under `data/processed/` so the notebook can start from an already-built feature table when desired.


## 1. Setup

Adjust the run flags below depending on whether you want a quick reproducible run or a full training run. The full SVD training uses more than 57 million ratings, so it is intentionally disabled by default.


In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
LIKE_THRESHOLD = 8.0
TOP_K = 10

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ARTIFACT_DIR = Path("artifacts")
MODEL_DIR = Path("models")

ANIME_PATH = RAW_DIR / "anime.csv"
ANIME_TEST_PATH = RAW_DIR / "anime_test.csv"
RATING_PATH = RAW_DIR / "rating_complete.csv"
RATING_TEST_PATH = RAW_DIR / "rating_test.csv"
META_PREPROCESSED_PATH = PROCESSED_DIR / "meta_preprocessed.csv"
META_READY_PATH = PROCESSED_DIR / "meta_train_ready.csv"
ENCODER_PATH = ARTIFACT_DIR / "tfidf_and_encoders.pkl"

REBUILD_PREPROCESSING = False
RUN_SVD_BASELINE = False
RUN_META_TRAINING = True
RUN_VISUALIZATION = False
RUN_INTERPRETATION = False

ARTIFACT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
required_files = [
    ANIME_PATH,
    ANIME_TEST_PATH,
    RATING_PATH,
    RATING_TEST_PATH,
    META_PREPROCESSED_PATH,
    META_READY_PATH,
]

for path in required_files:
    size_mb = path.stat().st_size / (1024 ** 2) if path.exists() else 0
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7} {path.as_posix():45} {size_mb:9.1f} MB")


## 2. Data Inspection

The main metadata table is `anime.csv`. The large rating tables are only previewed here to avoid loading the full dataset before a training cell actually needs it.


In [ ]:
anime_df = pd.read_csv(ANIME_PATH)
anime_test_df = pd.read_csv(ANIME_TEST_PATH)

print(f"anime.csv shape: {anime_df.shape}")
print(f"anime_test.csv shape: {anime_test_df.shape}")
print("\nColumns:")
print(anime_df.columns.tolist())

anime_df.head()


In [ ]:
print("Missing ratio, top 12 columns:")
display(anime_df.isna().mean().sort_values(ascending=False).head(12))

print("\nAnime type distribution:")
display(anime_df["Type"].value_counts(dropna=False).head(10))

print("\nRating table preview:")
display(pd.read_csv(RATING_PATH, nrows=5))

print("\nExternal rating test preview:")
display(pd.read_csv(RATING_TEST_PATH, nrows=5))


## 3. Metadata Preprocessing

The content-based module represents each anime using text features and numeric metadata:

- TF-IDF over `Genres`, `Producers`, and `Studios`.
- Label encoding for categorical fields.
- MinMax scaling for numeric popularity and score fields.

By default this notebook loads the precomputed file. Set `REBUILD_PREPROCESSING = True` to regenerate it.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler


def _clean_feature_name(prefix: str, name: str) -> str:
    safe = "".join(ch if ch.isalnum() or ch in ["_", "-"] else "_" for ch in str(name).lower())
    return f"{prefix}_{safe}".replace("__", "_").strip("_")


def make_tfidf_features(series: pd.Series, prefix: str, max_features: int = 100):
    vectorizer = TfidfVectorizer(
        token_pattern=r"[^, ]+",
        lowercase=True,
        max_features=max_features,
    )
    matrix = vectorizer.fit_transform(series.fillna("").astype(str))
    columns = [_clean_feature_name(prefix, name) for name in vectorizer.get_feature_names_out()]
    return pd.DataFrame(matrix.toarray(), columns=columns), vectorizer


def build_meta_preprocessed(anime: pd.DataFrame) -> pd.DataFrame:
    meta = anime.copy()
    meta = meta.fillna({
        "Genres": "Unknown",
        "Producers": "Unknown",
        "Studios": "Unknown",
        "Licensors": "Unknown",
        "Type": "Unknown",
        "Source": "Unknown",
        "Rating": "Unknown",
        "Premiered": "Unknown",
        "Duration": "Unknown",
    })

    meta["MAL_ID"] = meta["MAL_ID"].astype(int)

    numeric_cols = [
        "Score", "Episodes", "Ranked", "Popularity", "Members", "Favorites",
        "Watching", "Completed", "On-Hold", "Dropped", "Plan to Watch",
        "Score-10", "Score-9", "Score-8", "Score-7", "Score-6", "Score-5",
        "Score-4", "Score-3", "Score-2", "Score-1",
    ]
    numeric_cols = [col for col in numeric_cols if col in meta.columns]
    for col in numeric_cols:
        meta[col] = pd.to_numeric(meta[col].replace("Unknown", np.nan), errors="coerce")
        meta[col] = meta[col].fillna(meta[col].median())

    label_cols = ["Type", "Source", "Rating", "Premiered", "Duration"]
    label_cols = [col for col in label_cols if col in meta.columns]
    label_encoders = {}
    encoded = pd.DataFrame(index=meta.index)
    for col in label_cols:
        encoder = LabelEncoder()
        encoded[f"{col}_encoded"] = encoder.fit_transform(meta[col].astype(str))
        label_encoders[col] = encoder

    genre_df, vec_genres = make_tfidf_features(meta["Genres"], "Genre")
    producer_df, vec_producers = make_tfidf_features(meta["Producers"], "Prod")
    studio_df, vec_studios = make_tfidf_features(meta["Studios"], "Studio")

    scaler = MinMaxScaler()
    scaled_numeric = pd.DataFrame(
        scaler.fit_transform(meta[numeric_cols]),
        columns=numeric_cols,
        index=meta.index,
    )

    processed = pd.concat(
        [meta[["MAL_ID"]].reset_index(drop=True),
         scaled_numeric.reset_index(drop=True),
         encoded.reset_index(drop=True),
         genre_df.reset_index(drop=True),
         producer_df.reset_index(drop=True),
         studio_df.reset_index(drop=True)],
        axis=1,
    )

    processed.to_csv(META_PREPROCESSED_PATH, index=False)
    with open(ENCODER_PATH, "wb") as f:
        pickle.dump({
            "label_encoders": label_encoders,
            "vec_genres": vec_genres,
            "vec_producers": vec_producers,
            "vec_studios": vec_studios,
            "scaler": scaler,
            "numeric_cols": numeric_cols,
        }, f)
    return processed


if REBUILD_PREPROCESSING or not META_PREPROCESSED_PATH.exists():
    meta_preprocessed = build_meta_preprocessed(anime_df)
else:
    meta_preprocessed = pd.read_csv(META_PREPROCESSED_PATH)

print(f"meta_preprocessed shape: {meta_preprocessed.shape}")
meta_preprocessed.head()


## 4. SVD Baseline

This section trains the collaborative-filtering baseline and evaluates RMSE on `rating_test.csv`. It is disabled by default because the full training table is large.


In [ ]:
if RUN_SVD_BASELINE:
    try:
        from surprise import Dataset, Reader, SVD, accuracy
    except ImportError as exc:
        raise ImportError("Install scikit-surprise before running the SVD baseline.") from exc

    train_df = pd.read_csv(RATING_PATH)
    test_df = pd.read_csv(RATING_TEST_PATH)

    train_df = train_df[train_df["rating"] > 0]
    test_df = test_df[test_df["rating"] > 0]

    print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

    reader = Reader(rating_scale=(1, 10))
    train_data = Dataset.load_from_df(train_df[["user_id", "anime_id", "rating"]], reader)
    trainset = train_data.build_full_trainset()
    testset = list(zip(test_df["user_id"], test_df["anime_id"], test_df["rating"]))

    svd_model = SVD(
        n_factors=100,
        n_epochs=20,
        lr_all=0.005,
        reg_all=0.02,
        random_state=RANDOM_STATE,
        verbose=True,
    )
    svd_model.fit(trainset)
    predictions = svd_model.test(testset)
    svd_rmse = accuracy.rmse(predictions, verbose=True)

    with open(MODEL_DIR / "svd_model.pkl", "wb") as f:
        pickle.dump(svd_model, f)
else:
    print("RUN_SVD_BASELINE is False. Skipping full SVD training.")


## 5. Hybrid Meta-Training Data

`meta_train_ready.csv` contains the supervised training table for the meta learner:

- `cf_score`: SVD prediction.
- `cb_score`: content-based score.
- `true_rating`: ground-truth user rating.

This lets the final model learn a nonlinear combination of CF and CB signals.


In [ ]:
meta_ready = pd.read_csv(META_READY_PATH)
print(f"meta_train_ready shape: {meta_ready.shape}")
display(meta_ready.head())

display(meta_ready[["cf_score", "cb_score", "true_rating"]].describe())


## 6. Meta Learner Comparison

The report selected LightGBM as the strongest final model because it handled the compact continuous feature space well. This cell compares LightGBM with XGBoost and CatBoost when the libraries are installed.


In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import joblib


def rmse_score(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def precision_recall_at_k(frame: pd.DataFrame, pred_col: str, k: int = TOP_K, like_th: float = LIKE_THRESHOLD):
    precisions, recalls = [], []
    for _, group in frame.groupby("user_id"):
        ranked = group.sort_values(pred_col, ascending=False).head(k)
        relevant = group[group["true_rating"] >= like_th]
        if relevant.empty:
            continue
        hits = (ranked["true_rating"] >= like_th).sum()
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))
    return float(np.mean(precisions)), float(np.mean(recalls)), len(precisions)


if RUN_META_TRAINING:
    X = meta_ready[["cf_score", "cb_score"]]
    y = meta_ready["true_rating"]
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    models = {}

    try:
        import lightgbm as lgb
        models["LightGBM"] = lgb.LGBMRegressor(
            n_estimators=400,
            learning_rate=0.03,
            num_leaves=31,
            random_state=RANDOM_STATE,
        )
    except ImportError:
        print("LightGBM is not installed; skipping.")

    try:
        import xgboost as xgb
        models["XGBoost"] = xgb.XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RANDOM_STATE,
        )
    except ImportError:
        print("XGBoost is not installed; skipping.")

    try:
        from catboost import CatBoostRegressor
        models["CatBoost"] = CatBoostRegressor(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=0,
        )
    except ImportError:
        print("CatBoost is not installed; skipping.")

    results = []
    val_frame = meta_ready.loc[X_val.index, ["user_id", "anime_id", "true_rating"]].copy()

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        val_frame[f"pred_{name}"] = pred
        precision, recall, user_count = precision_recall_at_k(val_frame, f"pred_{name}")
        results.append({
            "model": name,
            "rmse": rmse_score(y_val, pred),
            f"precision@{TOP_K}": precision,
            f"recall@{TOP_K}": recall,
            "evaluated_users": user_count,
        })

    if results:
        results_df = pd.DataFrame(results).sort_values("rmse")
        display(results_df)
        best_name = results_df.iloc[0]["model"]
        best_model = models[best_name]
        joblib.dump(best_model, MODEL_DIR / "best_meta_model.pkl")
        print(f"Saved best model: {best_name} -> {MODEL_DIR / 'best_meta_model.pkl'}")
    else:
        print("No meta learner libraries are installed. Install requirements.txt and rerun this cell.")
else:
    print("RUN_META_TRAINING is False. Skipping meta learner comparison.")


## 7. Similar Anime Recommendation

This recommendation function finds users who strongly liked a target anime, scores their other watched anime with the selected meta model, and returns the highest-scoring titles.


In [ ]:
def find_anime_id(title: str, anime: pd.DataFrame = anime_df):
    matches = anime[anime["Name"].str.contains(title, case=False, na=False, regex=False)]
    if matches.empty:
        raise ValueError(f"No anime title found for: {title}")
    return int(matches.iloc[0]["MAL_ID"]), matches.iloc[0]["Name"]


def recommend_similar_by_model(title: str, model, top_n: int = 10):
    target_id, matched_title = find_anime_id(title)
    liked_users = meta_ready.loc[
        (meta_ready["anime_id"] == target_id) & (meta_ready["true_rating"] >= LIKE_THRESHOLD),
        "user_id",
    ].unique()

    if len(liked_users) == 0:
        raise ValueError(f"No high-rating users found for: {matched_title}")

    candidates = meta_ready[
        (meta_ready["user_id"].isin(liked_users)) &
        (meta_ready["anime_id"] != target_id)
    ].copy()
    candidates["predicted_rating"] = model.predict(candidates[["cf_score", "cb_score"]])

    ranked = (
        candidates.groupby("anime_id")
        .agg(mean_predicted_rating=("predicted_rating", "mean"), user_count=("user_id", "nunique"))
        .reset_index()
        .sort_values(["mean_predicted_rating", "user_count"], ascending=False)
        .head(top_n)
    )

    names = anime_df[["MAL_ID", "Name", "Genres", "Score"]].rename(columns={"MAL_ID": "anime_id"})
    ranked = ranked.merge(names, on="anime_id", how="left")
    ranked["mean_predicted_rating"] = ranked["mean_predicted_rating"].clip(1, 10)
    print(f"Recommendations conditioned on '{matched_title}' (MAL_ID={target_id})")
    return ranked[["anime_id", "Name", "Genres", "Score", "mean_predicted_rating", "user_count"]]


if "best_model" in globals():
    display(recommend_similar_by_model("Koe no Katachi", best_model, top_n=10))
else:
    print("Train or load a meta model first, then run recommend_similar_by_model(...).")


## 8. Optional Visualization

Set `RUN_VISUALIZATION = True` to project anime metadata embeddings into 3D with PCA and t-SNE. This is useful for presentation, but is not required for the core recommendation pipeline.


In [ ]:
if RUN_VISUALIZATION:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler
    import plotly.express as px

    merged = meta_preprocessed.merge(
        anime_df[["MAL_ID", "Name", "Genres", "Type"]], on="MAL_ID", how="left"
    )
    feature_cols = [col for col in meta_preprocessed.columns if col != "MAL_ID"]
    sample = merged.sample(min(6000, len(merged)), random_state=RANDOM_STATE)

    X_embed = StandardScaler(with_mean=False).fit_transform(sample[feature_cols])
    pca_dim = min(50, X_embed.shape[1])
    X_pca = PCA(n_components=pca_dim, random_state=RANDOM_STATE).fit_transform(X_embed)
    X_tsne = TSNE(n_components=3, random_state=RANDOM_STATE, init="pca", learning_rate="auto").fit_transform(X_pca)

    plot_df = sample[["Name", "Genres", "Type"]].copy()
    plot_df[["x", "y", "z"]] = X_tsne
    fig = px.scatter_3d(plot_df, x="x", y="y", z="z", color="Type", hover_name="Name", hover_data=["Genres"])
    fig.show()
else:
    print("RUN_VISUALIZATION is False. Skipping t-SNE visualization.")


## 9. Optional Interpretation

Set `RUN_INTERPRETATION = True` after training a LightGBM model to inspect feature influence with SHAP.


In [ ]:
if RUN_INTERPRETATION:
    if "best_model" not in globals():
        raise RuntimeError("Train a meta model before running interpretation.")

    import shap
    import matplotlib.pyplot as plt

    X_meta = meta_ready[["cf_score", "cb_score"]]
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_meta)
    shap.summary_plot(shap_values, X_meta)
else:
    print("RUN_INTERPRETATION is False. Skipping SHAP interpretation.")
